In [ ]:
%matplotlib inline
import torch
import torchvision
from torch.utils import data
from torchvision import transforms
from IPython import display

from matplotlib_inline import backend_inline
import matplotlib.pyplot as plt


In [100]:
import IPython
print(IPython.__version__) # 作为jupyter的依赖自动安装好了

9.15.0


In [101]:
def get_dataloader_workers():
    """使用4个进程读取数据"""
    return 0

In [102]:
# 整合所有的组件, 获取FashionMNIST数据集，返回训练集和验证集的数据迭代器

def load_data_fashion_mnist(batch_size, resize=None):
    """下载FashionMNIST的数据集，然后将其加载到内存中"""
    trans = [transforms.ToTensor()]
    if resize:
        trans.insert(0, transforms.Resize(resize))
    trans = transforms.Compose(trans)
    mnist_train = torchvision.datasets.FashionMNIST(
        root="../data", train=True, transform=trans, download=True
    ) 
    mnist_test = torchvision.datasets.FashionMNIST(
        root="../data", train=False, transform=trans, download=True
    ) 

    return (data.DataLoader(mnist_train, batch_size, shuffle=True, 
                num_workers=get_dataloader_workers()),
            data.DataLoader(mnist_train, batch_size, shuffle=True,
                num_workers=get_dataloader_workers()))

In [103]:
batch_size = 32
train_iter, test_iter = load_data_fashion_mnist(batch_size, resize=None)

In [104]:
num_inputs = 784 # 28 * 28
num_outputs = 10

In [105]:
display.display("hello")

'hello'

In [106]:
W = torch.normal(0, 0.01, size=(num_inputs, num_outputs), requires_grad=True)
b = torch.zeros(num_outputs, requires_grad=True)
W, W.shape, b, b.shape

(tensor([[ 8.5111e-04, -2.3844e-03, -1.0151e-02,  ..., -7.5443e-03,
           1.0338e-02, -7.5159e-05],
         [ 9.7790e-03, -2.3090e-02, -1.4298e-03,  ...,  1.7051e-02,
          -4.7266e-04, -9.3866e-04],
         [ 7.3990e-03, -1.0595e-02, -1.4333e-03,  ..., -4.9094e-03,
           6.9352e-03,  9.8888e-03],
         ...,
         [-7.2605e-03,  1.5028e-02, -8.7573e-03,  ..., -2.7741e-03,
          -9.0464e-03,  4.6312e-03],
         [ 2.7476e-03,  8.6602e-03, -1.5487e-02,  ...,  1.2731e-02,
           4.0677e-03,  1.6267e-02],
         [-1.9617e-02,  1.0634e-02, -2.2365e-04,  ...,  1.0999e-02,
          -1.0366e-03, -6.9804e-03]], requires_grad=True),
 torch.Size([784, 10]),
 tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True),
 torch.Size([10]))

In [107]:
# 定义softmax操作

X = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
X.sum(0, keepdim=True), X.sum(1, keepdim=True)

(tensor([[5., 7., 9.]]),
 tensor([[ 6.],
         [15.]]))

In [108]:
def softmax(X):
    X_exp = torch.exp(X)
    partition = X_exp.sum(1, keepdim=True)
    return X_exp / partition # 这里使用了广播机制

In [109]:
X = torch.normal(0, 1, (2, 5))
print(X)
print()
X_prob = softmax(X)
print(X_prob, X_prob.sum(1), sep='\n\n')

tensor([[-1.1174, -1.3726,  0.3508, -1.2364, -2.6626],
        [-0.9649,  0.3948, -0.1146, -0.9812,  0.6772]])

tensor([[0.1385, 0.1073, 0.6015, 0.1230, 0.0295],
        [0.0747, 0.2910, 0.1748, 0.0735, 0.3860]])

tensor([1., 1.])


In [110]:
def net(X):
    return softmax(torch.matmul(X.reshape(-1, W.shape[0]), W) + b)

In [111]:
# 定义损失函数
y = torch.tensor([0, 2])
print(y)
print()
y_hat = torch.tensor([[0.1, 0.3, 0.6],[0.3, 0.2, 0.5]])
print(y_hat[[0, 1], y])
print()
print(y_hat[:, y]) # 这俩还有区别
# x[[0,2], [1,0]] 配对取 (0,1)、(2,0)

tensor([0, 2])

tensor([0.1000, 0.5000])

tensor([[0.1000, 0.6000],
        [0.3000, 0.5000]])


In [112]:
def cross_entropy(y_hat, y):
    return -torch.log(y_hat[range(0, len(y_hat)), y])

cross_entropy(y_hat, y)

tensor([2.3026, 0.6931])

In [113]:
def accuracy(y_hat, y):
    """计算预测正确的数量"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1: 
        # 确保第一轴长度大于1，第二轴长度大于1
        y_hat = y_hat.argmax(axis = 1) # 得到每次预测的最大的概率的值的索引
    cmp = y_hat.type(y.dtype) == y
    # print(cmp)
    return float(cmp.type(y.dtype).sum())

In [114]:
accuracy(y_hat, y) / len(y)

0.5

In [115]:
class Accumulator:
    """在n个变量上累加"""
    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [116]:
# 评估任意模型的精度

def evaluate_accuracy(net, data_iter):
    """计算指定数据集上模型的精度"""
    if isinstance(net, torch.nn.Module):
        net.eval() # 将模型设置为评估模式
    metric = Accumulator(2) #正确预测数，预测总数，一个累加器类
    with torch.no_grad():
        for X, y in data_iter:
            metric.add(accuracy(net(X), y), y.numel())
            # accuracy 函数返回正确预测的数量, y.numel保存预测的数目

    return metric[0] / metric[1]

In [117]:
evaluate_accuracy(net, test_iter)

0.024183333333333334

In [119]:
# 训练
def train_epoch_ch3(net, train_iter, loss, updater):
    """训练模型一轮"""
    # 将模型设置为训练模式
    if isinstance(net, torch.nn.Module):
        net.train()

    metirc = Accumulator(3) # 训练损失总和, 训练准确度总和，样本数

    for X, y in train_iter:
        # 计算梯度并更新参数
        y_hat = net(X)
        l = loss(y_hat, y)

        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.mean().backward()
            updater.step()
        else:
            l.sum().backward()
            updater(X.shape[0])

        metirc.add(float(l.sum()), accuracy(y_hat, y), y.numel())

    # 返回损失和训练精度
    return metirc[0] / metirc[2], metirc[1] / metirc[2]
    

In [ ]:
class Animator: 
    """在动画中绘制数据"""
    def __init__(self, 
                 xlabel=None, 
                 ylabel=None, 
                 legend=None, 
                 xlim=None, 
                 ylim=None, 
                 xscale='linear', 
                 yscale='linear',
                 fmts=('-','m--','g-.','r:'),
                 nrows=1,
                 ncols=1,
                 figsize=(3.5, 2.5)):
        # 增量地绘制多条线
        if legend is None:
            legend = []

        backend_inline.select_figure_formats("svg")

        self.fig, self.ax = plt.subplots(nrows, ncols, figsize=figsize)

        if nrows * ncols == 1:
            self.axes = [self.axes, ]

        self.xlabel = xlabel
        self.ylabel = ylabel
        self.legend = legend
        self.xlim = xlim
        self.ylim = ylim
        self.xscale = xscale
        self.yscale = 
        

